In [ ]:

import pandas as pd

# 读取训练数据
train_file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv'
train_data = pd.read_csv(train_file_path)

# 查看数据的前几行
train_data.head()


Displayed on the user's machine.

In [ ]:

# 检查数据的缺失值
train_data.isnull().sum()


id               0
bone_length      0
rotting_flesh    0
hair_length      0
color            0
type             0
dtype: int64

In [ ]:


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 对类别标签进行编码
label_encoder_color = LabelEncoder()
train_data['color'] = label_encoder_color.fit_transform(train_data['color'])

label_encoder_type = LabelEncoder()
train_data['type'] = label_encoder_type.fit_transform(train_data['type'])

# 查看编码后的数据
train_data.head()



Displayed on the user's machine.

In [ ]:


from sklearn.preprocessing import StandardScaler

# 选择数值特征
numeric_features = ['bone_length', 'rotting_flesh', 'hair_length', 'color']

# 初始化标准化器
scaler = StandardScaler()

# 对数值特征进行标准化
train_data[numeric_features] = scaler.fit_transform(train_data[numeric_features])

# 查看标准化后的数据
train_data.head()



Displayed on the user's machine.

In [ ]:



# 选择特征和目标变量
X = train_data[numeric_features]
y = train_data['type']

# 划分数据集为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 查看训练集和测试集的形状
X_train.shape, X_test.shape, y_train.shape, y_test.shape




((236, 4), (60, 4), (236,), (60,))

In [ ]:



from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 选择随机森林分类器
model = RandomForestClassifier(random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 预测测试集
y_pred = model.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
accuracy



0.6166666666666667

In [ ]:



from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 初始化网格搜索
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, n_jobs=-1)

# 使用网格搜索进行训练
grid_search.fit(X_train, y_train)

# 获取最佳模型
best_model = grid_search.best_estimator_

# 使用最佳模型预测测试集
y_pred_best = best_model.predict(X_test)

# 计算最佳模型的准确率
best_accuracy = accuracy_score(y_test, y_pred_best)
best_accuracy




0.6166666666666667

In [ ]:



from sklearn.svm import SVC

# 选择支持向量机分类器
svm_model = SVC(random_state=42)

# 训练模型
svm_model.fit(X_train, y_train)

# 预测测试集
y_pred_svm = svm_model.predict(X_test)

# 计算准确率
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_accuracy




0.65

In [ ]:



from sklearn.linear_model import LogisticRegression

# 选择逻辑回归分类器
logreg_model = LogisticRegression(random_state=42, max_iter=1000)

# 训练模型
logreg_model.fit(X_train, y_train)

# 预测测试集
y_pred_logreg = logreg_model.predict(X_test)

# 计算准确率
logreg_accuracy = accuracy_score(y_test, y_pred_logreg)
logreg_accuracy




0.6666666666666666

In [ ]:




from xgboost import XGBClassifier

# 选择XGBoost分类器
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')

# 训练模型
xgb_model.fit(X_train, y_train)

# 预测测试集
y_pred_xgb = xgb_model.predict(X_test)

# 计算准确率
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_accuracy





D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [10:07:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
0.6

In [ ]:




from sklearn.ensemble import VotingClassifier

# 创建不同的分类器
classifiers = [
    ('logreg', LogisticRegression(random_state=42, max_iter=1000)),
    ('svm', SVC(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
]

# 选择投票分类器
voting_model = VotingClassifier(estimators=classifiers, voting='hard')

# 训练模型
voting_model.fit(X_train, y_train)

# 预测测试集
y_pred_voting = voting_model.predict(X_test)

# 计算准确率
voting_accuracy = accuracy_score(y_test, y_pred_voting)
voting_accuracy





0.6666666666666666

In [ ]:




# 添加交互特征
train_data['bone_rotting'] = train_data['bone_length'] * train_data['rotting_flesh']
train_data['bone_hair'] = train_data['bone_length'] * train_data['hair_length']
train_data['rotting_hair'] = train_data['rotting_flesh'] * train_data['hair_length']

# 选择新的特征
new_features = ['bone_length', 'rotting_flesh', 'hair_length', 'color', 'bone_rotting', 'bone_hair', 'rotting_hair']

# 重新划分数据集
X_new = train_data[new_features]
y_new = train_data['type']

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42)

# 查看新的训练集和测试集的形状
X_train_new.shape, X_test_new.shape, y_train_new.shape, y_test_new.shape





((236, 7), (60, 7), (236,), (60,))

In [ ]:





# 重新训练逻辑回归模型
logreg_model_new = LogisticRegression(random_state=42, max_iter=1000)

# 训练模型
logreg_model_new.fit(X_train_new, y_train_new)

# 预测测试集
y_pred_logreg_new = logreg_model_new.predict(X_test_new)

# 计算准确率
logreg_accuracy_new = accuracy_score(y_test_new, y_pred_logreg_new)
logreg_accuracy_new






0.6666666666666666

In [ ]:





from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# 定义基模型
base_estimators = [
    ('logreg', LogisticRegression(random_state=42, max_iter=1000)),
    ('svm', SVC(random_state=42, probability=True)),
    ('rf', RandomForestClassifier(random_state=42))
]

# 定义最终模型
final_estimator = LogisticRegression(random_state=42, max_iter=1000)

# 选择堆叠分类器
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=final_estimator)

# 训练模型
stacking_model.fit(X_train_new, y_train_new)

# 预测测试集
y_pred_stacking = stacking_model.predict(X_test_new)

# 计算准确率
stacking_accuracy = accuracy_score(y_test_new, y_pred_stacking)
stacking_accuracy






0.7166666666666667